# 问题——法律信息抽取

**学习目标**

1. 熟练掌握通过结构化提示词抽取信息
2. 学会 PDF 内容抽取与处理
3. 运用多线程提升批量处理效率

**作业说明**

1. 本作业需要同学补全少量代码，以使功能正常运行
2. 请在注释 #### 并标注下划线__________的地方补全相关代码。注释 # 的位置为普通注释，无需补全
3. 运行 Jupyter Notebook 全部代码块后，将文件导出为 HTML

## 任务1：设计结构化提示词

### 1.1 理解结构化提示词

**什么是结构化提示词？**  

结构化提示词是将提示词进一步组织、细化，按照一定的逻辑和格式来提供给AI。它的目的是让AI更清晰地理解你要表达的内容，进而生成更加精准和有用的结果。结构化提示词通常包括多个具体的元素，比如任务类型、要求、细节等。例如，如果你想让AI写一封邮件，而不仅仅是“写邮件”，你可以给出一个更结构化的提示词：


  “请帮我写一封邮件，内容如下：  
  收件人：老板  
  主题：下周会议安排  
  正文：尊敬的老板，您好！请问下周的会议安排如何？是否有需要提前准备的材料？感谢您的回复。  
  要求：语气要礼貌且专业，简洁明了。”  

这样的提示词比单纯的“写一封邮件”要具体和有条理，AI根据这些信息生成的邮件也会更符合你的期望。

**为什么结构化提示词很重要？**

1. 提高准确性：结构化提示词让AI能更精确地理解任务的各个细节，从而生成更符合要求的结果。例如，你提供具体的格式要求，AI就能在生成内容时遵循这些规则。  
2. 节省时间和精力：通过使用结构化提示词，你不需要反复修正AI的输出，能够更快速地得到满意的结果。你提供的信息越具体，AI的回答就越精准，减少了你对结果的修改和调整。  
3. 减少模糊性：一般的提示词容易让AI产生模糊的理解，而结构化提示词能清晰界定每个部分的信息要求，避免了不必要的误解。

**示例**

```python
prompt_template = '''
提取以下内容中的法律要素：
{content}

输出格式：
{
  "被告": "",
  "诉讼请求": "",
  "判决结果": ""
}
'''

**为什么要这么设计？**

1. 明确任务：我们首先清楚地告诉 AI 任务是什么：“提取以下内容中的法律要素”。这样，AI 能准确理解任务目标，避免产生不相关的输出。
2. 灵活性：使用占位符{content}，可以动态插入任何法律文本。这让模板通用，适用于不同文本。
3. 结构化输出：规定了输出格式（如“被告”: “”），确保 AI 返回一致、规范、非随机的数据，方便后续处理。
4. 高效处理：结构化输出易于机器或人类直接利用，减少了额外解析工作，提高了工作效率。

总结来说，这样的设计帮助 AI 更精准地完成任务，同时让输出更有用、易操作。

### 1.2 不同提示词的效果对比

**选取一篇判决书作为示例**

In [51]:
legal_doc = "浙江省绍兴县人民法院民事判决书（2007）绍民二初字第1348号原告吴江德英化工有限公司。法定代表人杨照生。委托代理人（特别授权代理）申铁旗。被告绍兴县华正纺织品整理有限公司。法定代表人吴建华。被告杜方稳。两被告的委托代理人（特别授权代理）吴红英。原告吴江德英化工有限公司诉被告华正公司、杜方稳买卖合同纠纷一案，本院于2007年7月30日立案受理，依法由审判员张关雄适用简易程序于同年8月20日公开开庭进行了审理。原告的委托代理人申铁旗，两被告的委托代理人吴红英到庭参加诉讼。本案现已审理终结。原告诉称，原告与两被告有业务往来关系。2005年9月15日，两被告向原告购买硬胶和软胶，合计货款15,444元。两被告收到货物后一直未付款，经原告多次催讨，均无果。要求判令两被告支付货款15,444元，本案诉讼费由被告负担。被告华正公司辩称，华正公司没有委托被告杜方稳与原告签订购销合同，也没有收到原告的货物。要求驳回原告的诉讼请求。被告杜方稳辩称，与原告签订购销合同事实，但没有收到合同中约定的货物。要求驳回原告的诉讼请求。原告为证明其主张，在庭审中向本院举证如下：1、吴晓萍的名片一份，证明吴晓萍是被告华正公司职工，吴晓萍打电话给原告要求购买硬胶、软胶，并指定原告把货交给杜方稳的事实；2、购销合同（纸张为绿颜色）一份，以证明原告向两被告提供货物的名称，被告杜方稳代表被告华正公司在该购销合同兼送货单上签字的事实。对原告的举证，被告华正公司质证认为，原告证1，没有委托吴晓萍向原告购买货物，且被告华正公司根本没有吴晓萍此人。原告证2，被告华正公司没有收到该货物，也没有委托被告杜方稳签收货物。被告杜方稳质证认为，原告证1，有否此事不清楚。原告证2，没有收到合同中的货物，如被告杜方稳已收货，原告应当提供红色的提货单。两被告未提供证据。经原告举证、被告质证，本院认证如下，原告证1，吴晓萍的名片不能反映出吴晓萍打电话给原告的情况，故不予认定。原告证2，购销合同的供方为原告，需方为华正公司，纸张为绿色，纸张的右边标有“①白存根②红提货③绿回单”的字样。该合同约定了标的物的品名、数量、单价、金额、交货日期、交货地点、质量问题的处理方式等内容，被告杜方稳在该合同需方处签了名，但该合同内容中，没有被告杜方稳已收到合同标的物的意思表示，故该购销合同不能作为认定被告杜方稳收取原告所供货物的事实依据。综上，本院对本案事实作如下认定：2005年9月15日，原告与被告杜方稳签订购销合同一份。合同约定，原告供需方被告华正公司硬胶、软胶，金额为15,444元。合同还对标的物的规格、数量、单价、交货日期、交货地点、质量问题的处理方式等作了约定。该合同纸张为绿色，纸张的右边标有“①白存根②红提货③绿回单”的字样。被告杜方稳在该合同需方处签了名。嗣后，原告要求被告华正公司、被告杜方稳支付上述合同标的物的货款，两被告未予支付，原告故提起诉讼。本院认为，2005年9月15日，原告与被告杜方稳签订购销合同的事实清楚，证据确实。原告诉称两被告已收取了该合同约定的硬胶、软胶，故要求两被告支付货款，但原告对此主张未能举证证明，本院不予采信。鉴此，依照《中华人民共和国民事诉讼法》第六十四条、《中华人民共和国合同法》第五条之规定，判决如下：驳回原告吴江德英化工有限公司的诉讼请求。案件受理费186元，减半收取93元，由原告负担。如不服本判决，可在判决书送达之日起15日内，向本院递交上诉状，并按对方当事人的人数提出副本，上诉于浙江省绍兴市中级人民法院（在递交上诉状之日起7日内先预缴上诉案件受理费186元，款汇绍兴市预算外资金财政专户，账号：09×××27，开户行：绍兴市商业银行业务部。逾期按自动撤回上诉处理）。审判员 张关雄二〇〇七年九月三日书记员 徐 芳 来源：百度搜索“马克数据网”"

**普通提示词**

f""" 是一种格式化字符串的方式，在 Python 中用来将变量嵌入到字符串中。通过这种方式，你可以在字符串中直接插入变量值，非常简洁和高效

{legal_doc} 是我们刚才定义的文书，它可以代表任何你传给它的法律文书内容。使用 f"""，我们可以直接把 legal_doc 的内容嵌入到提示词里。这意味着我们能够根据不同的输入动态生成提示词，而不需要手动拼接每次的文本

In [53]:
normal_prompt =  f"""这里是个法律文书
{legal_doc}，告诉我里面的被告，诉讼和判决结果 """

**半结构化提示词**

In [55]:
semistructured_prompt = f"""
请解析以下法律文书：
{legal_doc}

输出：
- 被告信息
- 诉讼请求
- 判决结果
"""

**进阶版结构化提示词**

In [57]:
structured_prompt = f"""
请严格按JSON格式解析内容：
{legal_doc}

输出结构：
{{
  "案件编号": "YYYY-XXXX格式",
  "被告信息": {{
    "姓名": "",
    "辩护人": ""
  }},
  "诉讼请求": ["列表形式"],
  "判决结果": {{
    "时间": "YYYY-MM-DD",
    "金额": 数值,
    "法律依据": ["《...》第X条"]
  }}
}}
"""

**为了方便后续的调用，我们将 api_key, base_url, 以及需要用到的模型存在变量中**

In [193]:
api_key = ______________________  # 填写 api_key
base_url = "https://open.bigmodel.cn/api/paas/v4/"
model = "glm-4-flash"

**创建对话函数**

In [195]:
from zhipuai import ZhipuAI

# 创建 ZhipuAI 实例
client = ZhipuAI(
    api_key=api_key,
    base_url=base_url
)

#### 定义问答函数
def get_response(prompt):
    response = ______________________
    return response.______________________

**生成回答结果**

In [59]:
normal_output = ______________________   #### 补全代码，按普通提示词模板生成回答
basic_output = ______________________    #### 补全代码，按半结构化提示词模板生成回答
structured_output = ______________________    #### 补全代码，，按结构化提示词模板生成回答

**结果对比展示**

In [64]:
from IPython.display import display, Markdown

display(Markdown("**普通提示词输出：**"))
print(normal_output)

display(Markdown("**半结构化提示词输出：**"))
print(basic_output)

display(Markdown("**结构化提示词输出：**"))
print(structured_output)

**普通提示词输出：**

根据您提供的法律文书内容，以下是被告、诉讼和判决结果的具体信息：

**被告**：
1. 绍兴县华正纺织品整理有限公司（法定代表人：吴建华）
2. 杜方稳

**诉讼**：
原告吴江德英化工有限公司诉被告绍兴县华正纺织品整理有限公司、杜方稳买卖合同纠纷一案。

**判决结果**：
浙江省绍兴县人民法院判决如下：
- 驳回原告吴江德英化工有限公司的诉讼请求。
- 案件受理费186元，减半收取93元，由原告负担。

该判决书表明，法院认为原告未能提供充分的证据证明被告已收取了合同约定的货物，因此驳回了原告要求被告支付货款的诉讼请求。


**基础提示词输出：**

- 被告信息：
  1. 被告绍兴县华正纺织品整理有限公司，法定代表人吴建华。
  2. 被告杜方稳。

- 诉讼请求：
  原告吴江德英化工有限公司要求判令被告华正公司和杜方稳支付货款15,444元，本案诉讼费由被告负担。

- 判决结果：
  浙江省绍兴县人民法院判决驳回原告吴江德英化工有限公司的诉讼请求。案件受理费186元，减半收取93元，由原告负担。


**结构化提示词输出：**

```json
{
  "案件编号": "2007-绍民二初字第1348号",
  "被告信息": {
    "姓名": "绍兴县华正纺织品整理有限公司, 杜方稳",
    "辩护人": "吴红英"
  },
  "诉讼请求": [
    "要求判令两被告支付货款15,444元",
    "本案诉讼费由被告负担"
  ],
  "判决结果": {
    "时间": "2007-09-03",
    "金额": 93,
    "法律依据": [
      "《中华人民共和国民事诉讼法》第六十四条",
      "《中华人民共和国合同法》第五条"
    ]
  }
}
```


**结构化提示词更容易让我们在后续对数据进行批量化处理**

## 任务2：PDF 内容抽取

PDF 文件中的数据通常是非结构化的，这意味着它们呈现的是一种平面的文本格式，缺乏明确的字段分隔和标签。然而，很多分析任务（比如法律文本分析、数据提取等）要求我们从这些非结构化数据中提取出特定的信息（比如被告信息、诉讼请求、判决结果等）。通过将 PDF 中的内容转化为结构化数据（如 JSON 格式），我们能更方便地进行后续分析和处理

### 2.1 通过 PyPDF2 解析文件


**PyPDF2 是一个免费的、开源的纯 Python 库，能够拆分、合并、裁剪和转换PDF文件的页面**

In [ ]:
pip install PyPDF2

In [191]:
# 导入 PyPDF2 库，用于处理 PDF 文件
import PyPDF2

# 定义一个函数，接收 PDF 文件路径作为输入，返回提取的文本
def pdf_to_text(file_path):
    # 初始化一个空字符串，用于存储提取到的文本
    text = ""
    
    # 使用 'with open' 打开文件，这样文件用完后会自动关闭
    with open(file_path, "rb") as f:
        # 创建一个 PdfReader 对象，用于读取 PDF 文件
        reader = PyPDF2.PdfReader(f)
        
        # 遍历 PDF 中的每一页
        for page in reader.pages:
            # 从每一页中提取文本，并将其添加到 'text' 字符串中
            text += page.extract_text() + "\n"  # 每一页的文本后加一个换行符
    
    # 返回提取到的文本内容
    return text

In [27]:
# 实战测试部分：指定一个 PDF 文件的路径
sample_pdf = "./法律判决书/2007绍民二初字第1348号.pdf"

#### 调用 pdf_to_text 函数，提取 PDF 中的文本，并打印前 500 个字
print(______________________)

浙江省绍兴县人民法院民 事 判 决 书（2007）绍民二初字第1348号原告吴江德英化工有限公
司。法定代表人杨照生。委托代理人（特别授权代理）申铁旗。被告绍兴县华正纺织品整理有
限公司。法定代表人吴建华。被告杜方稳。两被告的委托代理人（特别授权代理）吴红英。原
告吴江德英化工有限公司诉被告华正公司、杜方稳买卖合同纠纷一案，本院于2007年7月30日
立案受理，依法由审判员张关雄适用简易程序于同年8月20日公开开庭进行了审理。原告的委
托代理人申铁旗，两被告的委托代理人吴红英到庭参加诉讼。本案现已审理终结。原告诉称，
原告与两被告有业务往来关系。2005年9月15日，两被告向原告购买硬胶和软胶，合计货款15
，444元。两被告收到货物后一直未付款，经原告多次催讨，均无果。要求判令两被告支付货
款15，444元，本案诉讼费由被告负担。被告华正公司辩称，华正公司没有委托被告杜方稳与
原告签订购销合同，也没有收到原告的货物。要求驳回原告的诉讼请求。被告杜方稳辩称，与
原告签订购销合同事实，但没有收到合同中约定的货物。要求驳回原告的诉讼请求。原告为证
明其主张，在庭审中向本院举证如下：1、吴晓


### 2.2 信息提取方法对比

**在先前的作业中，我们曾经使用 ZhipuAI 类中的 files.content 方法来抽取信息，下面我们比较一下两种方法**

**首先回顾通过 files.content 抽取信息的 parse_file 函数**

In [44]:
from pathlib import Path
import json

#### 生成 ZhipuAI 实例，传入两个参数 api_key 和 base_url
client = ______________________

# 定义信息抽取函数
def parse_file(file_path):
    # 使用 Path 类将文件路径字符串转换为 Path 对象
    # 将文件上传到智谱 AI 开放平台，返回一个文件对象
    file_object = client.files.create(file=Path(file_path), purpose="file-extract")

    # 文件内容抽取，包含 content，file_type，filename，title，type 五个字段
    file_content = client.files.content(file_id=file_object.id).content.decode()

    # 建议在提取数据后删除文件，并将文件抽取内容存储到本地，避免重复上传。
    client.files.delete(file_id=file_object.id)

    #### 补全代码，返回文档内容（content），需要转为 JSON 格式
    return json.______________________

**实现方式对比**

- parse_file 方法：
借助智谱 AI 开放平台提供的 API 进行文件内容抽取。首先将文件上传到平台，获取文件对象，然后从平台获取文件内容，最后将文件从平台删除。
依赖外部平台，需要与平台进行网络交互，上传和下载文件内容。
- pdf_to_text 方法：
使用 Python 的 PyPDF2 库在本地直接处理 PDF 文件。通过 PdfReader 对象逐页读取 PDF 文件，并提取文本内容。
完全在本地完成操作，不依赖外部服务。

**性能对比**

In [70]:
#### 补全代码，测试 parse_file 方法（上传智谱开放平台解析）的提取速度
import time

start_time = ______________________
parse_file("./法律判决书/2007绍民二初字第1348号.pdf")
end_time = ______________________
time_spent = ______________________
print('开放平台方法耗时：', ______________________, '秒')

开放平台方法耗时： 0.8900418281555176 秒


In [72]:
#### 参考上方实现方法，测试 pdf_to_text 方法（本地处理）的提取速度

______________________
print('本地方法耗时：', time_spent, '秒')

本地方法耗时： 0.040242910385131836 秒


**优先使用 parse_file 方法的场景**

```#### 补全场景总结，从性能、返回信息类型、文件格式、数据安全角度辨析```

```______________________```

**优先使用 pdf_to_text 方法的场景**

```#### 补全场景总结，从性能、返回信息类型、文件格式、数据安全角度辨析```

```______________________```

## 任务3：运用多线程批量处理文件

### 3.1 并发与多线程的基本概念

**并发**

并发是一种处理多个任务的方式，它允许程序在同一时间段内处理多个任务。注意，并发并不一定意味着这些任务是同时执行的。在单核 CPU 系统中，并发是通过快速地在多个任务之间切换来实现的，给人一种多个任务同时执行的错觉。想象你是一个餐厅的服务员，同一时间有好几桌客人都有需求。你没办法同时给所有客人服务，但你可以在给一桌客人点菜的间隙，去另一桌给客人倒杯水，然后再回来接着点菜。虽然同一时刻你只能服务一桌客人，但通过快速地在不同客人之间切换，你感觉好像同时在为很多桌客人服务

而在多核 CPU 系统中，并发可以真正实现多个任务的同时执行


**多线程**

多线程是实现并发的一种具体方式。线程是程序执行的最小单位，一个进程可以包含多个线程。多线程允许程序在同一进程内创建多个线程，这些线程可以并发执行不同的任务

在 Python 中，多线程对于 I/O 密集型任务非常有用，因为在 I/O 操作时，线程会释放 GIL（全局解释器锁），让其他线程有机会执行。但对于 CPU 密集型任务，由于 GIL 的存在，多线程并不能真正实现并行计算，反而可能因为线程切换带来额外的开销


**concurrent.futures 和 ThreadPoolExecutor**

concurrent.futures 是 Python 3.2 引入的一个高级模块，它提供了一个简单的接口来实现异步执行任务。该模块包含两个主要的类：ThreadPoolExecutor 和 ProcessPoolExecutor，分别用于实现多线程和多进程

ThreadPoolExecutor 是一个线程池执行器，它可以管理一个线程池，负责创建、管理和销毁线程。使用线程池的好处是可以避免频繁创建和销毁线程带来的开销，提高程序的性能

线程池就好比餐厅里固定的一组服务员，每次有客人（任务）来的时候，不用临时去招聘新的服务员，直接从这组服务员里选一个去服务客人就行。任务完成后，这个服务员也不会被辞退，而是回到团队里等待下一个任务。这样可以避免频繁招聘和辞退服务员（创建和销毁线程）带来的麻烦和成本

**示例：计算一个数的 2 倍**

In [104]:
from concurrent.futures import ThreadPoolExecutor

def task_function(arg):
    # 这里是具体的任务逻辑
    print(f"Processing {arg}")
    return arg * 2

# 任务参数列表
task_args = [1, 2, 3, 4, 5]

# 创建一个最大线程数为 5 的线程池执行器。with 语句会在代码块结束时自动关闭线程池，释放资源
with ThreadPoolExecutor(max_workers=5) as executor:
    # 用于将任务提交到线程池执行。submit 方法会返回一个 Future 对象，代表一个异步执行的任务
    futures = [executor.submit(task_function, arg) for arg in task_args]

    # 获取任务结果
    for future in futures:
        result = future.result()
        print(f"Result: {result}")

Processing 1
Processing 2
Processing 3
Result: 2
Result: 4
Result: 6
Processing 4
Processing 5
Result: 8
Result: 10


### 3.2 比较多线程和单线程抽取 PDF 文件信息的效率

In [110]:
!pip install zhipuai gradio langchain langchain-community pyMuPDF tqdm pandas openpyxl ace_tools

In [306]:
# 导入所需的库
import os  # 用于文件和目录的操作
import time  # 用于计时
import concurrent.futures  # 用于并发处理任务
from tqdm import tqdm  # 用于显示进度条
import pandas as pd  # 用于数据处理，特别是表格数据
import PyPDF2  # 用于处理 PDF 文件
from zhipuai import ZhipuAI  # 用于大模型能力调用

# 指定路径
legal_docs = "./法律判决书"  # PDF文件的存放路径

**定义信息抽取函数**

In [304]:
def Chat(text):
    """
    处理单个文档的核心逻辑，主要用于从法律文书中提取关键信息。

    :param text: 法律文书的文本内容，但在当前函数中未使用该参数，可根据实际情况后续调整
    :return: 若API请求成功，返回模型生成的包含关键信息的JSON格式字符串；若请求失败，返回空字典
    """
    # 构建提示词，告知模型从法律文书中提取特定的关键信息，并规定返回的JSON格式
    prompt = f"""
    请从以下法律文书中提取关键信息：    
    要求返回JSON格式：
    {{
        "案件编号": "格式：YYYY-编号",  # 法律案件的编号，格式为年份加上编号
        "诉讼金额": 纯数字,  # 该法律案件涉及的诉讼金额，仅包含数字
        "判决日期": "YYYY-MM-DD",  # 法律案件的判决日期，格式为年-月-日
        "审理法院": ""  # 负责审理该法律案件的法院名称
    }}
    """ + text
    
    try:
        # 调用API，向指定的模型发送请求，以获取关键信息
        chat = client.chat.completions.create(
            model=model,  # 指定使用的模型名称
            messages=[{"role": "user", "content": prompt}],  # 发送用户的提示词作为输入
        )
        # 从API响应中提取模型生成的结果并返回
        return chat.choices[0].message.content
    except Exception as e:
        # 若API请求过程中出现异常，打印详细的错误信息
        print(f"API请求失败：{str(e)}")
        # 返回空字典，表示请求失败，未获取到有效信息
        return {}

**定义单线程处理函数**

In [302]:
def single_thread_processing(file_list):
    """
    单线程处理文件列表，依次对列表中的每个文件进行处理。

    :param file_list: 待处理的文件路径列表，列表中的每个元素为一个文件的路径
    :return: 包含每个文件处理结果的列表，处理结果由 Chat 函数生成
    """
    # 初始化一个空列表，用于存储每个文件的处理结果
    results = []
    # 遍历文件列表，对每个文件进行顺序处理
    # 使用 tqdm 库显示处理进度条，方便用户了解处理进度
    for file_path in tqdm(file_list, desc="单线程处理"):
        # 调用 pdf_to_text 函数从 PDF 文件中提取文本内容
        text = pdf_to_text(file_path)
        # 检查是否成功提取到文本内容
        if text:
            # 如果提取到文本，调用 Chat 函数处理文本并将结果添加到结果列表中
            results.append(Chat(text))
    # 返回包含所有文件处理结果的列表
    return results

**定义多线程处理函数**

In [300]:
def multi_thread_processing(file_list, workers=5):
    """
    多线程处理文件列表，通过创建线程池并发处理文件，以加速处理过程。

    :param file_list: 待处理的文件路径列表，列表中的每个元素为一个文件的路径
    :param workers: 线程池中的最大工作线程数，默认为 5
    :return: 包含每个文件处理结果的列表，处理结果由 Chat 函数生成
    """
    # 初始化一个空列表，用于存储每个文件的处理结果
    results = []

    def process_single_file(file_path):
        """
        处理单个文件的子任务，包括从文件中提取文本并进行分析。

        :param file_path: 待处理的单个文件的路径
        :return: 如果成功提取到文本，返回 Chat 函数处理后的结果；否则返回空字典
        """
        #### 调用 pdf_to_text 函数从 PDF 文件中提取文本内容
        text = ______________________
        
        # 检查是否成功提取到文本内容
        # 如果提取到文本，调用 Chat 函数处理文本；否则返回空字典
        return Chat(text) if text else {}

    # 使用 ThreadPoolExecutor 创建一个线程池，指定最大工作线程数
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        # 为文件列表中的每个文件提交处理任务到线程池
        #### 补全代码，创建一个字典，键为多线程执行 process_single_file 的 Future 对象，值为对应的文件路径
        futures = {______________________ for fp in file_list}

        # 使用 tqdm 库显示并发处理的进度条
        # concurrent.futures.as_completed 函数会在任务完成时返回对应的 Future 对象
        for future in tqdm(concurrent.futures.as_completed(futures),
                           total=len(file_list),
                           desc="多线程处理"):
            # 获取每个任务的处理结果，并添加到结果列表中
            results.append(future.result())
    # 返回包含所有文件处理结果的列表
    return results

**获取文件列表，列出所有 PDF 文件**

In [298]:
def get_files(legal_docs, num_files=None):
    """
    获取文件列表，从指定目录中获取所有PDF文件，并可选择返回指定数量的文件。

    :param legal_docs: PDF文件所在目录路径
    :param num_files: 要获取的文件数量，如果为 None 则获取所有文件
    :return: 包含PDF文件路径的列表
    """
    all_files = [os.path.join(legal_docs, f)
                 for f in os.listdir(legal_docs)
                 if f.endswith(".pdf")]
    if num_files is not None:
        return all_files[:num_files]
    return all_files

**定义性能对比实验函数，可以自定义文件个数**

In [296]:
# 性能对比实验
def performance_comparison(legal_docs, num_files=None):
    """
    进行单线程和多线程处理的性能对比实验。

    :param legal_docs: PDF文件所在目录路径
    :param num_files: 要处理的文件数量，如果为 None 则处理所有文件
    """
    test_files = get_files(legal_docs, num_files)

    # 单线程处理测试
    start = time.time()  # 记录开始时间
    single_results = single_thread_processing(test_files)  # 调用单线程处理函数处理文件
    single_time = time.time() - start  # 计算单线程处理的耗时

    # 多线程处理测试
    start = time.time()  # 记录开始时间
    multi_results = multi_thread_processing(test_files)  # 调用多线程处理函数处理文件
    multi_time = time.time() - start  # 计算多线程处理的耗时

    # 结果对比：使用 pandas 将单线程和多线程处理的结果整理为表格
    df = pd.DataFrame({
        "处理方式": ["单线程处理", "多线程处理（5线程）"],  # 显示不同的处理方式
        "总耗时（秒）": [single_time, multi_time],  # 总耗时
        "平均耗时/文件（秒）": [single_time / len(test_files), multi_time / len(test_files)],  # 平均耗时/文件
        "吞吐量（文件/秒）": [len(test_files) / single_time, len(test_files) / multi_time]  # 吞吐量
    })

    print("\n性能对比结果：")  # 输出对比结果
    display(df.style
          .format("{:.2f}", subset=["总耗时（秒）", "平均耗时/文件（秒）", "吞吐量（文件/秒）"]))  # 格式化输出

    return multi_results

**比较抽取 10 个文件的效率**

In [244]:
num_files_to_test = 10

#### 补全代码，比较单线程和多线程处理前 num_files_to_test 个文件的效率
results = ______________________

多线程处理: 100%|██████████| 10/10 [00:14<00:00,  1.48s/it]


性能对比结果：


,处理方式,总耗时（秒）,平均耗时/文件（秒）,吞吐量（文件/秒）
0,单线程处理,35.67,3.57,0.28
1,多线程处理（5线程）,14.86,1.49,0.67


### 3.3 格式整理

**目标：将结果转换为标准的 Pandas DataFrame**

**观察结果返回的格式**

In [246]:
results

['```json\n{\n    "案件编号": "2007-219",\n    "诉讼金额": 0,\n    "判决日期": "2007-08-02",\n    "审理法院": "浙江省淳安县人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2007-1408",\n    "诉讼金额": 20000,\n    "判决日期": "2007-08-20",\n    "审理法院": "浙江省绍兴县人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2001-新法执字第61号",\n    "诉讼金额": null,  # 文书中未提及具体的诉讼金额\n    "判决日期": "1999-12-02",  # 原判决日期\n    "审理法院": "西安市新城区人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2007-越民二初字第1370号",\n    "诉讼金额": 33800,\n    "判决日期": "2007-07-30",\n    "审理法院": "浙江省绍兴市越城区人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2007-武侯行初字第27号",\n    "诉讼金额": null,\n    "判决日期": "2007-09-19",\n    "审理法院": "成都市武侯区人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2007-雁民初字第1878号",\n    "诉讼金额": 250,\n    "判决日期": "2007-08-01",\n    "审理法院": "西安市雁塔区人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2007-1985",\n    "诉讼金额": 76924.33,\n    "判决日期": "2007-07-18",\n    "审理法院": "浙江省绍兴县人民法院"\n}\n```',
 '```json\n{\n    "案件编号": "2007-1235",\n    "诉讼金额": 27000,\n    "判决日期": "2007-09-11",\n    "审理法院": "浙

**问题1. 生成结果两端包含 Markdown 标识 ```json**

**问题2. 部分生成结果中包含注释，例如：**

 '```json\n{\n    "案件编号": "2001-新法执字第61号",\n    "诉讼金额": null,  # 文书中未提及具体的诉讼金额\n    "判决日期": "1999-12-02",  # 原判决日期\n    "审理法院": "西安市新城区人民法院"\n}\n```'

**问题3. result 可能为空字典（不为字符串），因为我们先前定义了API 和文件解析shi'bai返回结果**

**因此需要解决格式问题以保证后续的格式转换**

In [294]:
# 使用正则表达式匹配格式
import re

# 解析 JSON 数据并存储到列表中
data_list = []
for result in results:
    try:
        # 尝试将 result 当作字符串处理，否则说明 API 请求失败返回了空字典
        # 去除 Markdown 代码块标记
        cleaned_json_str = result.replace("```json", "").replace("```", "").strip()
        # 去除 JSON 字符串中的注释
        cleaned_json_str = re.sub(r'#.*', '', cleaned_json_str)
    except AttributeError:
        print(f"API 请求失败")   
    
    try:
        # 解析 JSON 数据
        json_data = json.loads(cleaned_json_str)
        data_list.append(json_data)
    except json.JSONDecodeError as e:
        print(f"无法解析 JSON 数据: {e}，请检查字符串格式。")

**查看结果**

In [ ]:
# 创建 DataFrame
df = pd.DataFrame(data_list)

# 查看数据表格
df

### 3.4 实战环节：抽取所有法律信息

In [292]:
#### 定义新函数 legal_information_extraction
def legal_information_extraction(legal_docs, num_files=None):
    """
    使用多线程处理抽取所有文件的法律信息，对结果进行格式清洗后写入 Excel 文件。

    :param legal_docs: PDF 文件所在目录路径
    :param num_files: 要处理的文件数量，如果为 None 则处理所有文件
    """
    # 获取文件列表
    target_files = ______________________

    # 使用多线程处理抽取所有文件信息
    results = ______________________

    # 存储解析后的数据
    data_list = []

    # 遍历结果列表，对每个结果进行格式清洗和解析
    for result in results:
        try:
            # 尝试将 result 当作字符串处理，否则说明 API 请求失败返回了空字典
            # 去除 Markdown 代码块标记
            cleaned_json_str = result.replace("```json", "").replace("```", "").strip()
            # 去除 JSON 字符串中的注释
            cleaned_json_str = re.sub(r'#.*', '', cleaned_json_str)
        except AttributeError:
            print(f"API 请求失败")        
        
        try:
            # 解析 JSON 数据
            json_data = json.loads(cleaned_json_str)
            data_list.append(json_data)
        except json.JSONDecodeError as e:
            print(f"无法解析 JSON 数据: {e}，请检查字符串格式。")

    # 创建 DataFrame
    df = ______________________

    # 定义导出文件名
    excel_file_path = 'legal_information.xlsx'
    
    #### 写入 Excel，不设 index
    ______________________

    print(f"数据已成功写入 {excel_file_path}")

**执行信息抽取检查结果**

- 预计处理时长半个小时
- 中间可能遇到部分因内容安全审核导致的 API 请求失败和 JSON 格式错误，忽略即可

In [314]:
legal_information_extraction(legal_docs)

多线程处理:   1%|          | 30/2997 [00:21<21:05,  2.34it/s] 

API请求失败：Error code: 400, with error text {"contentFilter":[{"level":1,"role":"user"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


多线程处理:   6%|▋         | 191/2997 [02:01<30:46,  1.52it/s]

API请求失败：Error code: 400, with error text {"contentFilter":[{"level":1,"role":"user"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


多线程处理:  54%|█████▎    | 1610/2997 [14:46<08:20,  2.77it/s] 

API请求失败：Error code: 400, with error text {"contentFilter":[{"level":1,"role":"user"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


多线程处理: 100%|██████████| 2997/2997 [27:21<00:00,  1.83it/s]


无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 24 (char 55)，请检查字符串格式。
API 请求失败
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 20 (char 52)，请检查字符串格式。
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 21 (char 54)，请检查字符串格式。
API 请求失败
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 21 (char 53)，请检查字符串格式。
无法解析 JSON 数据: Expecting ',' delimiter: line 3 column 17 (char 48)，请检查字符串格式。
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 20 (char 51)，请检查字符串格式。
无法解析 JSON 数据: Expecting ',' delimiter: line 3 column 18 (char 49)，请检查字符串格式。
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 20 (char 51)，请检查字符串格式。
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 20 (char 51)，请检查字符串格式。
无法解析 JSON 数据: Expecting property name enclosed in double quotes: line 3 column 18 (char 49)，请检查字符串格式。
无法解析 JSON 数据: 